# PyTorch Basics: Autograd - Automatic Differentiation

Welcome to the second PyTorch tutorial! This notebook covers:
- What is automatic differentiation?
- Computational graphs
- Gradient computation with `.backward()`
- Gradient accumulation and zeroing
- Disabling gradient tracking

## What is Autograd?

**Autograd** is PyTorch's automatic differentiation engine. It powers neural network training by:
- Automatically computing gradients
- Building computational graphs dynamically
- Enabling backpropagation

```
Forward Pass          Backward Pass
────────────────────────────────────
x → f(x) → y        y → ∂y/∂x → x.grad
```

The key idea: PyTorch tracks all operations on tensors that have `requires_grad=True` and can automatically compute gradients.


In [2]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")


PyTorch version: 2.13.0


## 1. Basic Gradient Computation

Let's start with a simple example: computing the gradient of y = x²


In [3]:
# Create a tensor with gradient tracking enabled
x = torch.tensor(3.0, requires_grad=True)
print(f"x = {x}")
print(f"x.requires_grad = {x.requires_grad}")
print()

# Forward pass: compute y = x²
y = x ** 2
print(f"y = x² = {y}")
print(f"y.requires_grad = {y.requires_grad}")
print()

# Backward pass: compute gradient dy/dx
y.backward()

# The gradient dy/dx = 2x is stored in x.grad
print(f"dy/dx = 2x = 2*3 = {x.grad}")
print(f"Computed gradient: {x.grad}")


x = 3.0
x.requires_grad = True

y = x² = 9.0
y.requires_grad = True

dy/dx = 2x = 2*3 = 6.0
Computed gradient: 6.0


## 2. Computational Graphs

PyTorch builds a dynamic computational graph during the forward pass.

```
Example: z = (x + y) * (y + 2)

    x          y
     \        / \
      \      /   \
       \    /     \
        add      add
         \       /
          \     /
           \   /
            mul
             |
             z

Each operation creates a node in the graph.
Gradients flow backwards through this graph.
```


In [4]:
# More complex example
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Forward pass
z = x**2 + y**3
print(f"x = {x}, y = {y}")
print(f"z = x² + y³ = {z}")
print()

# Backward pass
z.backward()

# Gradients
print(f"∂z/∂x = 2x = 2*2 = {x.grad}")
print(f"∂z/∂y = 3y² = 3*9 = {y.grad}")


x = 2.0, y = 3.0
z = x² + y³ = 31.0

∂z/∂x = 2x = 2*2 = 4.0
∂z/∂y = 3y² = 3*9 = 27.0


## 3. Vector and Matrix Gradients


In [5]:
# Vector example
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
z = y.sum()  # Must reduce to scalar for backward()

print(f"x = {x}")
print(f"y = x² = {y}")
print(f"z = sum(y) = {z}")
print()

z.backward()
print(f"∂z/∂x = 2x = {x.grad}")
print()

# Matrix example
X = torch.randn(2, 3, requires_grad=True)
Y = X * 2
Z = Y.mean()

print(f"X shape: {X.shape}")
print(f"Z = {Z}")

Z.backward()
print(f"X.grad shape: {X.grad.shape}")
print(f"X.grad:\n{X.grad}")


x = tensor([1., 2., 3.], requires_grad=True)
y = x² = tensor([1., 4., 9.], grad_fn=<PowBackward0>)
z = sum(y) = 14.0

∂z/∂x = 2x = tensor([2., 4., 6.])

X shape: torch.Size([2, 3])
Z = -2.05505633354187
X.grad shape: torch.Size([2, 3])
X.grad:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])


## 4. Gradient Accumulation

⚠️ **Important**: Gradients accumulate by default!


In [6]:
x = torch.tensor(3.0, requires_grad=True)

# First computation
y = x ** 2
y.backward()
print(f"After first backward: x.grad = {x.grad}")

# Second computation (gradient accumulates!)
y = x ** 3
y.backward()
print(f"After second backward: x.grad = {x.grad}")
print(f"Expected 3x² = {3 * x.item()**2}, but got accumulated value")
print()

# Solution: Zero gradients before each backward pass
x.grad.zero_()
y = x ** 3
y.backward()
print(f"After zeroing and backward: x.grad = {x.grad}")


After first backward: x.grad = 6.0
After second backward: x.grad = 33.0
Expected 3x² = 27.0, but got accumulated value

After zeroing and backward: x.grad = 27.0


## 5. Disabling Gradient Tracking

Sometimes you don't need gradients (e.g., during inference).


In [7]:
x = torch.tensor(3.0, requires_grad=True)

# Method 1: torch.no_grad() context manager
with torch.no_grad():
    y = x ** 2
    print(f"Inside no_grad: y.requires_grad = {y.requires_grad}")

# Method 2: .detach()
y = (x ** 2).detach()
print(f"Using detach: y.requires_grad = {y.requires_grad}")

# Method 3: .requires_grad_(False)
x.requires_grad_(False)
y = x ** 2
print(f"After requires_grad_(False): y.requires_grad = {y.requires_grad}")

# Re-enable
x.requires_grad_(True)
y = x ** 2
print(f"After requires_grad_(True): y.requires_grad = {y.requires_grad}")


Inside no_grad: y.requires_grad = False
Using detach: y.requires_grad = False
After requires_grad_(False): y.requires_grad = False
After requires_grad_(True): y.requires_grad = True


## 6. Gradient of Non-Scalar Outputs

For non-scalar outputs, you need to provide a gradient argument.


In [8]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

print(f"y = {y}")
print(f"y is not a scalar, shape: {y.shape}")
print()

# Provide gradient for each output
gradient = torch.tensor([1.0, 1.0, 1.0])
y.backward(gradient=gradient, retain_graph=True)
print(f"x.grad with gradient=[1,1,1]: {x.grad}")
print()

# Reset
x.grad.zero_()
gradient = torch.tensor([0.1, 1.0, 100.0])
y.backward(gradient=gradient)
print(f"x.grad with gradient=[0.1,1,100]: {x.grad}")


y = tensor([1., 4., 9.], grad_fn=<PowBackward0>)
y is not a scalar, shape: torch.Size([3])

x.grad with gradient=[1,1,1]: tensor([2., 4., 6.])

x.grad with gradient=[0.1,1,100]: tensor([2.0000e-01, 4.0000e+00, 6.0000e+02])


## 7. Common Activation Functions and Their Gradients


In [9]:
import torch.nn.functional as F

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)

# ReLU
y = F.relu(x)
print(f"ReLU({x.data}) = {y}")
y.sum().backward()
print(f"ReLU gradient: {x.grad}")
print()

# Sigmoid
x.grad.zero_()
y = torch.sigmoid(x)
print(f"Sigmoid({x.data}) = {y}")
y.sum().backward()
print(f"Sigmoid gradient: {x.grad}")
print()

# Tanh
x.grad.zero_()
y = torch.tanh(x)
print(f"Tanh({x.data}) = {y}")
y.sum().backward()
print(f"Tanh gradient: {x.grad}")


ReLU(tensor([-2., -1.,  0.,  1.,  2.])) = tensor([0., 0., 0., 1., 2.], grad_fn=<ReluBackward0>)
ReLU gradient: tensor([0., 0., 0., 1., 1.])

Sigmoid(tensor([-2., -1.,  0.,  1.,  2.])) = tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808], grad_fn=<SigmoidBackward0>)
Sigmoid gradient: tensor([0.1050, 0.1966, 0.2500, 0.1966, 0.1050])

Tanh(tensor([-2., -1.,  0.,  1.,  2.])) = tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640], grad_fn=<TanhBackward0>)
Tanh gradient: tensor([0.0707, 0.4200, 1.0000, 0.4200, 0.0707])


## 8. Practical Example: Linear Regression

Let's implement gradient descent manually using autograd.


In [13]:
# Generate synthetic data: y = 3x + 2 + noise
torch.manual_seed(42)
X_train = torch.randn(100, 1) * 10
y_train = 3 * X_train + 2 + torch.randn(100, 1) * 2

# Initialize parameters
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01
epochs = 500

# Training loop
for epoch in range(epochs):
    # Forward pass
    y_pred = X_train * w + b
    
    # Compute loss (MSE)
    loss = ((y_pred - y_train) ** 2).mean()
    
    # Backward pass
    loss.backward()
    
    # Update parameters (no gradient tracking)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, w = {w.item():.4f}, b = {b.item():.4f}")

print(f"\nFinal parameters: w = {w.item():.4f}, b = {b.item():.4f}")
print(f"True parameters: w = 3.0, b = 2.0")


Epoch 20: Loss = 54.4278, w = 2.3478, b = 0.5496
Epoch 40: Loss = 7.3240, w = 2.8406, b = 1.0556
Epoch 60: Loss = 3.7998, w = 2.9640, b = 1.3926
Epoch 80: Loss = 3.3478, w = 2.9944, b = 1.6176
Epoch 100: Loss = 3.2177, w = 3.0015, b = 1.7680
Epoch 120: Loss = 3.1642, w = 3.0029, b = 1.8685
Epoch 140: Loss = 3.1405, w = 3.0030, b = 1.9357
Epoch 160: Loss = 3.1300, w = 3.0029, b = 1.9807
Epoch 180: Loss = 3.1253, w = 3.0027, b = 2.0107
Epoch 200: Loss = 3.1231, w = 3.0026, b = 2.0308
Epoch 220: Loss = 3.1222, w = 3.0025, b = 2.0442
Epoch 240: Loss = 3.1218, w = 3.0025, b = 2.0532
Epoch 260: Loss = 3.1216, w = 3.0024, b = 2.0592
Epoch 280: Loss = 3.1215, w = 3.0024, b = 2.0632
Epoch 300: Loss = 3.1215, w = 3.0024, b = 2.0659
Epoch 320: Loss = 3.1215, w = 3.0024, b = 2.0677
Epoch 340: Loss = 3.1214, w = 3.0024, b = 2.0689
Epoch 360: Loss = 3.1214, w = 3.0024, b = 2.0697
Epoch 380: Loss = 3.1214, w = 3.0024, b = 2.0703
Epoch 400: Loss = 3.1214, w = 3.0024, b = 2.0706
Epoch 420: Loss = 3.121

## 9. Higher-Order Gradients

You can compute gradients of gradients (second derivatives).


In [11]:
x = torch.tensor(2.0, requires_grad=True)

# First derivative
y = x ** 3
y.backward(create_graph=True)  # Keep graph for second derivative
print(f"y = x³")
print(f"dy/dx = 3x² = {x.grad}")

# Second derivative
x.grad.backward()
print(f"d²y/dx² = 6x = {x.grad}")  # This will show accumulated gradient

# Clean example
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
grad2 = torch.autograd.grad(grad1, x)[0]
print(f"\nUsing torch.autograd.grad:")
print(f"First derivative: {grad1}")
print(f"Second derivative: {grad2}")


y = x³
dy/dx = 3x² = 12.0
d²y/dx² = 6x = 24.0

Using torch.autograd.grad:
First derivative: 12.0
Second derivative: 12.0


/Users/nageshnazare/.pyenv/versions/3.14.0/lib/python3.14/site-packages/torch/autograd/graph.py:979: UserWarning: Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/engine.cpp:1312.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


## 📝 Summary

You've learned:

✅ What autograd is and how it works  
✅ Creating computational graphs  
✅ Computing gradients with `.backward()`  
✅ Gradient accumulation and zeroing  
✅ Disabling gradient tracking  
✅ Gradients for vectors and matrices  
✅ Implementing gradient descent manually  

### Key Points to Remember

1. **Always zero gradients** before backward pass: `x.grad.zero_()`
2. **Use torch.no_grad()** during inference to save memory
3. **Call .backward() only on scalars** (or provide gradient argument)
4. **Gradients accumulate** - this is a feature, not a bug!

### Next Steps

- **Next Tutorial**: `03_datasets.ipynb` - Learn about data loading
- **Practice**: Implement logistic regression using autograd
- **Challenge**: Implement a simple neural network from scratch

### Additional Resources

- [PyTorch Autograd Documentation](https://pytorch.org/docs/stable/autograd.html)
- [Autograd Mechanics](https://pytorch.org/docs/stable/notes/autograd.html)


## 🎯 Practice Exercises


In [14]:
# Exercise 1: Compute gradient of f(x) = sin(x²) at x = 1
# Your code here:
# Analytical derivative: f'(x) = 2x * cos(x²) -> f'(1) = 2 * cos(1) ≈ 1.0806
# =====================================================================
x = torch.tensor(1.0, requires_grad=True)
y = torch.sin(x ** 2)
y.backward()
print("Exercise 1:")
print(f"Computed gradient at x=1: {x.grad.item():.4f}")
print(f"Analytical gradient (2*cos(1)): {2 * torch.cos(torch.tensor(1.0)).item():.4f}")
print()

# Exercise 2: Implement gradient descent for y = ax² + bx + c
# Find a, b, c given data points
# Your code here:
torch.manual_seed(42)
# True parameters: a=2.0, b=-1.5, c=3.0
X = torch.linspace(-2, 2, 100).unsqueeze(1)
y_true = 2.0 * (X ** 2) - 1.5 * X + 3.0 + torch.randn(100, 1) * 0.1
# Initialize parameters
a = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
c = torch.randn(1, requires_grad=True)
learning_rate = 0.05
epochs = 500
for epoch in range(epochs):
    # Forward pass: quadratic model
    y_pred = a * (X ** 2) + b * X + c
    
    # Compute MSE loss
    loss = torch.mean((y_pred - y_true) ** 2)
    
    # Backward pass
    loss.backward()
    
    # Update weights without gradient tracking
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
    
    # Zero gradients for the next iteration
    a.grad.zero_()
    b.grad.zero_()
    c.grad.zero_()
print("Exercise 2:")
print(f"Estimated parameters: a = {a.item():.4f}, b = {b.item():.4f}, c = {c.item():.4f}")
print("Target parameters:    a = 2.0000, b = -1.5000, c = 3.0000")
print()


# Exercise 3: Compute the Jacobian matrix for a function f: R² → R²
# Your code here:
def f(x):
    return torch.stack([
        x[0]**2 + x[1],
        3 * x[0] + x[1]**3
    ])
input_tensor = torch.tensor([2.0, 3.0])
# Method 1: Using torch.autograd.functional.jacobian
jacobian_matrix = torch.autograd.functional.jacobian(f, input_tensor)
print("Exercise 3:")
print(f"Input point: {input_tensor.tolist()}")
print(f"Jacobian matrix:\n{jacobian_matrix}")
# Analytical Jacobian: [[2*x1, 1], [3, 3*x2²]] at [2, 3] -> [[4, 1], [3, 27]]
print()


# Exercise 4: Implement custom loss function and compute its gradient
# Your code here:
def custom_huber_loss(y_pred, y_target, delta=1.0):
    error = y_pred - y_target
    abs_error = torch.abs(error)
    # Quadratic for small errors, linear for large errors
    quadratic = torch.clamp(abs_error, max=delta)
    linear = abs_error - quadratic
    loss = 0.5 * (quadratic ** 2) + delta * linear
    return loss.mean()
y_pred = torch.tensor([2.5, 0.0, 8.0], requires_grad=True)
y_target = torch.tensor([3.0, -2.0, 4.0])
loss = custom_huber_loss(y_pred, y_target, delta=1.0)
loss.backward()
print("Exercise 4:")
print(f"Custom Huber Loss: {loss.item():.4f}")
print(f"Gradients on predictions (dLoss/dy_pred): {y_pred.grad}")

Exercise 1:
Computed gradient at x=1: 1.0806
Analytical gradient (2*cos(1)): 1.0806

Exercise 2:
Estimated parameters: a = 2.0014, b = -1.5015, c = 3.0041
Target parameters:    a = 2.0000, b = -1.5000, c = 3.0000

Exercise 3:
Input point: [2.0, 3.0]
Jacobian matrix:
tensor([[ 4.,  1.],
        [ 3., 27.]])

Exercise 4:
Custom Huber Loss: 1.7083
Gradients on predictions (dLoss/dy_pred): tensor([-0.1667,  0.3333,  0.3333])
